# 10. 패들 쉬프트 원본 대조와 표 청킹 보완

## 이번 단계에서 배우는 것
검색용 글에서 깨진 기호를 고치되 `raw_text`(PDF에서 추출한 원문)는 그대로 둡니다.
표는 **드라이브 모드 + 조작 기호 + 기능**을 한 행으로 묶습니다.
검색에서 작은 조각을 찾으면 부모 원문과 연결된 주의사항까지 읽도록 관계를 기록합니다.

PDF 397~398쪽(매뉴얼 392~393쪽), PDF 402쪽(매뉴얼 397쪽)을 페이지 전체 화면과 대조했습니다.
397쪽 `/I1`은 시동 경고 화면이고 `/I2`는 패들 쉬프트 그림임을 개별 이미지로 확인했습니다.

아래 셀을 위에서부터 실행하세요. 09번 전체 청킹 초안을 다시 만드는 첫 셀은 시간이 걸릴 수 있습니다.
이번 실험은 메모리에서 기록과 청크만 만들며, 데이터 파일·벡터·Supabase 저장은 하지 않습니다.


## 수업 기반과 프로젝트 추가 기록 (2026-10-01)

코드의 추가 설명은 주석으로 남겼습니다. 이 표시는 프로젝트 적용 기록이며, 수업에서 개념 자체를 다루지 않았다는 뜻은 아닙니다.

- 원본 화면에서 확인한 패들 쉬프트의 기호·표 행·그림을 보완합니다.
- 추가: 확인한 구간에만 î/ï 기호 수정, /I2 그림 선택, 모드별 행 보존, 조작·제한 조건 연결.
- 아직 대조하지 않은 403쪽은 미검토 자료로 남깁니다.

전체 목록: [수업 기반·추가 적용 기록](../src/zzong_santafe_lag/learning_additions.md)


In [1]:
# [프로젝트 추가 기록] 원본 화면에서 확인한 패들 쉬프트의 기호·표 행·그림을 보완합니다.
# 추가 이유·한계·수업 근거 전체 목록: src/zzong_santafe_lag/learning_additions.md
from pathlib import Path
import json
import contextlib
import io
from copy import deepcopy
from collections import Counter, defaultdict

project_folder = Path.cwd().parent if Path.cwd().name.lower() == "notebooks" else Path.cwd()
notebook_path = project_folder / "notebooks" / "09_full_manual_topic_preview.ipynb"

# 앞 실험의 함수와 자료를 재사용합니다. 나중에 이 코드를 .py 함수로 옮깁니다.
previous_notebook = json.loads(notebook_path.read_text(encoding="utf-8"))
draft = {}
with contextlib.redirect_stdout(io.StringIO()):
    for index, cell in enumerate(previous_notebook["cells"]):
        if cell["cell_type"] == "code":
            exec(compile("".join(cell["source"]), f"09:cell_{index}", "exec"), draft)

page_inventory = draft["page_inventory"]
reader = draft["reader"]
prepare_text = draft["prepare_text"]
make_search_chunks = draft["make_search_chunks"]
token_budget = draft["token_budget"]

# deepcopy는 복사본을 수정해도 09번 실험 자료가 바뀌지 않게 합니다.
original_parents = draft["full_parent_records"]
reviewed_parent_records = deepcopy(original_parents)
print("불러온 원문 묶음:", len(original_parents))
print("입력 한도:", token_budget, "토큰")


불러온 원문 묶음: 524
입력 한도: 128 토큰


## 1. 바꿀 구간만 원문에서 가져오기

원본 확인을 마친 402쪽과 아직 확인하지 않은 403쪽을 구분합니다.
원문 전체와 각 구간의 글자 위치를 유지하므로 나중에 출처를 다시 찾아갈 수 있습니다.


In [2]:
# [프로젝트 추가 기록] 원본 화면에서 확인한 패들 쉬프트의 기호·표 행·그림을 보완합니다.
# 추가 이유·한계·수업 근거 전체 목록: src/zzong_santafe_lag/learning_additions.md
def find_parent(title, first_page):
    """제목과 시작 PDF 쪽수가 일치하는 원문 묶음 한 개를 찾습니다."""
    matches = [
        parent for parent in reviewed_parent_records
        if parent["metadata"]["title"] == title
        and parent["metadata"]["pdf_page_number"] == first_page
    ]
    if len(matches) != 1:
        raise ValueError(f"{title}: 원문 묶음이 달라졌으므로 경계를 다시 확인하세요.")
    return matches[0]


def select_page_spans(parent, numbers, record_id):
    """선택한 페이지 구간만 복사하고, 원문 위치와 출처를 함께 유지합니다."""
    spans = [
        deepcopy(span) for span in parent["metadata"]["source_spans"]
        if span["pdf_page_number"] in numbers
    ]
    if not spans:
        raise ValueError("선택할 원문 구간이 없습니다.")
    pages = list(dict.fromkeys(span["pdf_page_number"] for span in spans))
    raw = "\n".join(
        page_inventory[span["pdf_page_number"]]["body_text"][span["start"]:span["end"]]
        for span in spans
    )
    record = {
        "content": prepare_text(raw), "raw_text": raw,
        "metadata": deepcopy(parent["metadata"]),
        "image_parts": [
            deepcopy(part) for part in parent["image_parts"]
            if part["pdf_page_number"] in pages
        ],
    }
    record["metadata"].update({
        "record_id": record_id, "source_pages": pages, "source_spans": spans,
        "pdf_page_number": pages[0],
        "manual_page_number": page_inventory[pages[0]]["manual_page_number"],
    })
    return record


paddle_draft = find_parent("패들 쉬프트 (수동 변속 모드)", 397)
mode_table_draft = find_parent("주행 모드별 패들 쉬프트의 기능", 402)
limit_draft = find_parent("회생 제동 시스템 사용 제한", 402)

paddle_record = select_page_spans(paddle_draft, {397}, paddle_draft["metadata"]["record_id"])
table_398_record = select_page_spans(paddle_draft, {398}, "reviewed_paddle_table_398")
table_398_record["metadata"]["title"] = "드라이브 모드 별 패들 쉬프트 레버 기능"
table_402_record = deepcopy(mode_table_draft)
limit_402_record = select_page_spans(limit_draft, {402}, limit_draft["metadata"]["record_id"])
limit_403_draft = select_page_spans(limit_draft, {403}, "unreviewed_paddle_limit_403")
limit_403_draft["metadata"]["title"] = "시스템 제한 경고"
limit_403_draft["metadata"]["verification_status"] = "auto_draft_needs_review"

# 목차 경계가 실제 화면의 제목 경계와 일치하는지 확인합니다.
assert paddle_record["raw_text"].startswith("패들 쉬프트 (수동 변속 모드)")
assert "올바른 운전 요령" not in table_398_record["raw_text"]
assert "회생 제동 시스템 사용 제한" not in table_402_record["raw_text"]
assert limit_402_record["raw_text"].startswith("회생 제동 시스템 사용 제한")
assert [page_inventory[n]["manual_page_number"] for n in [397, 398, 402]] == [392, 393, 397]
print("보완 대상:", [(r["metadata"]["title"], r["metadata"]["source_pages"])
    for r in [paddle_record, table_398_record, table_402_record, limit_402_record]])


보완 대상: [('패들 쉬프트 (수동 변속 모드)', [397]), ('드라이브 모드 별 패들 쉬프트 레버 기능', [398]), ('주행 모드별 패들 쉬프트의 기능', [402]), ('회생 제동 시스템 사용 제한', [402])]


## 2. 기호·그림·표 행을 보완하기

- `î → +`, `ï → -`는 이번에 화면으로 확인한 397쪽 구간에만 적용합니다.
- 397쪽 패들 쉬프트에는 `I2.jpg`만 연결합니다.
- 398쪽과 402쪽의 동일한 표는 출처별로 남기고 같은 표라는 표시를 붙입니다.
- SPORT의 기능은 원문처럼 '수동 변속 +'와 '수동 변속 -'로 기록합니다.


In [3]:
# [프로젝트 추가 기록] 원본 화면에서 확인한 패들 쉬프트의 기호·표 행·그림을 보완합니다.
# 추가 이유·한계·수업 근거 전체 목록: src/zzong_santafe_lag/learning_additions.md
# 전역 치환을 하면 다른 페이지의 기호까지 잘못 바꿀 수 있어 확인한 구간만 고칩니다.
symbol_corrections = {"î": "+", "ï": "-"}
corrected_raw = paddle_record["raw_text"]
for broken, corrected in symbol_corrections.items():
    corrected_raw = corrected_raw.replace(broken, corrected)
paddle_record["content"] = prepare_text(corrected_raw)

# 개별 이미지로 확인한 그림 하나만 선택합니다. 실제 이미지 파일은 저장하지 않습니다.
paddle_parts = [
    part for part in paddle_record["image_parts"] if part["pdf_image_key"] == "/I2"
]
if len(paddle_parts) != 1:
    raise ValueError("패들 쉬프트 이미지 키가 달라졌습니다.")
paddle_image = reader.pages[396].images["/I2"]
paddle_part = deepcopy(paddle_parts[0])
paddle_part.update({
    "name": paddle_image.name, "size": tuple(paddle_image.image.size),
    "description": "스티어링 휠 뒤쪽의 패들 쉬프트 레버와 당기는 방향을 보여준다.",
    "linkage_status": "visually_verified",
})
paddle_record["image_parts"] = [paddle_part]
paddle_record["content"] += "\n[그림 설명] " + paddle_part["description"]
paddle_record["metadata"]["image_linkage_status"] = "visually_verified"
paddle_record["metadata"]["symbol_corrections"] = symbol_corrections

# 표는 열 머리글과 셀의 관계를 확인한 후 행으로 옮깁니다.
table_rows = [
    {"drive_mode": "ECO", "lever": "+", "function": "회생 제동 단계 감소"},
    {"drive_mode": "ECO", "lever": "-", "function": "회생 제동 단계 증가"},
    {"drive_mode": "SPORT", "lever": "+", "function": "수동 변속 +"},
    {"drive_mode": "SPORT", "lever": "-", "function": "수동 변속 -"},
]


def normalize_mode_table(record):
    """확인한 네 표 행을 옮기고 표 앞 설명과 뒤 참고 문장을 유지합니다."""
    raw = record["raw_text"]
    start = raw.index("드라이브 모드 패들 쉬프트 레버 조작 패들 쉬프트 레버 기능")
    last_row = "- 수동 변속 -"
    end = raw.index(last_row, start) + len(last_row)
    extracted_rows = raw[start:end]
    # 같은 PDF가 바뀌면 예전에 확인한 표를 그대로 적용하지 않습니다.
    expected_rows = [
        "ECO", "+ 회생 제동 단계 감소", "- 회생 제동 단계 증가",
        "SPORT", "+ 수동 변속 +", "- 수동 변속 -",
    ]
    actual_rows = [line.strip() for line in extracted_rows.splitlines()[1:] if line.strip()]
    if actual_rows != expected_rows:
        raise ValueError("표 내용이 바뀌었습니다. 원본 화면부터 확인하세요.")

    row_lines = [
        f'드라이브 모드: {row["drive_mode"]} | 패들 쉬프트 레버 조작: {row["lever"]} | 기능: {row["function"]}'
        for row in table_rows
    ]
    record["content"] = "\n".join([
        prepare_text(raw[:start]), "[표: 드라이브 모드별 패들 쉬프트 레버 기능]",
        *row_lines, prepare_text(raw[end:]),
    ])
    record["metadata"].update({
        "content_type": "structured_table",
        "table_rows": deepcopy(table_rows),
        "duplicate_group_id": "paddle_mode_function_table",
        "table_structure_status": "visually_verified",
    })
    return record


table_398_record = normalize_mode_table(table_398_record)
table_402_record = normalize_mode_table(table_402_record)
limit_402_record["metadata"]["content_type"] = "instruction_text"

reviewed_records = [paddle_record, table_398_record, table_402_record, limit_402_record]
for record in reviewed_records:
    record["metadata"].update({
        "verification_status": "visually_reviewed_source",
        "review_method": "원본 페이지 전체 화면 및 표 행·기호 대조",
        "review_flags": [],
    })

# 연결은 검색 결과를 답변 근거로 읽을 때 함께 불러와야 할 원문 ID입니다.
# 이 표시만으로 기존 답변 코드가 자동으로 읽는 것은 아니므로 이후 검색 코드에 반영합니다.
paddle_id = paddle_record["metadata"]["record_id"]
table_398_id = table_398_record["metadata"]["record_id"]
table_402_id = table_402_record["metadata"]["record_id"]
limit_402_id = limit_402_record["metadata"]["record_id"]
paddle_record["metadata"]["required_context_record_ids"] = [table_398_id]
table_398_record["metadata"]["required_context_record_ids"] = [paddle_id, limit_402_id]
table_402_record["metadata"]["required_context_record_ids"] = [paddle_id, limit_402_id]
limit_402_record["metadata"]["required_context_record_ids"] = [table_402_id]
limit_402_record["metadata"]["unreviewed_continuation_record_ids"] = [
    limit_403_draft["metadata"]["record_id"]
]

# 해당하는 세 초안만 대체하고 나머지 기록은 그대로 남깁니다.
replacements = {
    paddle_draft["metadata"]["record_id"]: [paddle_record, table_398_record],
    mode_table_draft["metadata"]["record_id"]: [table_402_record],
    limit_draft["metadata"]["record_id"]: [limit_402_record, limit_403_draft],
}
reviewed_parent_records = [
    replacement
    for parent in reviewed_parent_records
    for replacement in replacements.get(parent["metadata"]["record_id"], [parent])
]
print("전체 원문 묶음:", len(reviewed_parent_records))
print("원본 대조를 마친 기록:", len(reviewed_records))
print("패들 쉬프트 연결 그림:", paddle_part["name"], paddle_part["size"])
print("\n402쪽 표 검색용 글:\n" + table_402_record["content"])


전체 원문 묶음: 526
원본 대조를 마친 기록: 4
패들 쉬프트 연결 그림: I2.jpg (687, 438)

402쪽 표 검색용 글:
주행 모드별 패들 쉬프트의 기능 [참고] 선택된 드라이브 모드에 따라 패들 쉬프트의 기능이 달라집니다.
[표: 드라이브 모드별 패들 쉬프트 레버 기능]
드라이브 모드: ECO | 패들 쉬프트 레버 조작: + | 기능: 회생 제동 단계 감소
드라이브 모드: ECO | 패들 쉬프트 레버 조작: - | 기능: 회생 제동 단계 증가
드라이브 모드: SPORT | 패들 쉬프트 레버 조작: + | 기능: 수동 변속 +
드라이브 모드: SPORT | 패들 쉬프트 레버 조작: - | 기능: 수동 변속 -
ECO 모드에서 시동을 껐다가 다시 걸면 ECO 모드가 유지됩니다. SPORT 모드에서 시동을 껐다가 다시 걸면 ECO 모드로 변경됩니다. 자세한 내용은 6장 내 ' 통합 주행 모드 시스템 (2WD) ' 을 참고하십시오.


## 3. 표의 한 행을 자르지 않는 검색 조각 만들기

이 모델의 입력 한도는 128토큰입니다. 토큰은 모델이 글을 읽을 때 사용하는 작은 단위입니다.
한 행에 모드·기호·기능을 함께 적고, 한도를 넘으면 무리하게 잘라 쓰지 않고 오류를 내도록 합니다.
설명문은 앞서 만든 문장 단위 분할 함수를 그대로 사용합니다.


In [4]:
# [프로젝트 추가 기록] 원본 화면에서 확인한 패들 쉬프트의 기호·표 행·그림을 보완합니다.
# 추가 이유·한계·수업 근거 전체 목록: src/zzong_santafe_lag/learning_additions.md
def make_reviewed_chunks(parent):
    """표는 행 단위, 설명문은 문장 단위로 토큰 한도 안에서 나눕니다."""
    if parent["metadata"]["content_type"] == "structured_table":
        # 05번 안내도 분할 함수는 한 줄을 보존하므로 확인한 표 행에도 재사용합니다.
        proxy = deepcopy(parent)
        proxy["metadata"]["content_type"] = "vehicle_overview_navigation"
        chunks = make_search_chunks(proxy, token_budget)
        for chunk in chunks:
            chunk["metadata"]["content_type"] = "structured_table"
            chunk["metadata"]["chunking_unit"] = "complete_table_row_or_context_line"
    else:
        chunks = make_search_chunks(parent, token_budget)
    for chunk in chunks:
        chunk["metadata"]["verification_status"] = parent["metadata"]["verification_status"]
        chunk["metadata"]["review_flags"] = parent["metadata"].get("review_flags", [])
        chunk["metadata"]["required_context_record_ids"] = parent["metadata"].get(
            "required_context_record_ids", []
        )
    return chunks


reviewed_search_chunks = [
    chunk for parent in reviewed_parent_records for chunk in make_reviewed_chunks(parent)
]
parent_lookup = {parent["metadata"]["record_id"]: parent for parent in reviewed_parent_records}
chunk_lookup = {chunk["metadata"]["record_id"]: chunk for chunk in reviewed_search_chunks}


def read_reviewed_context(chunk_id):
    """찾은 조각의 부모 원문과 연결된 확인 기록을 중복 없이 메모리에서 불러옵니다."""
    chunk = chunk_lookup[chunk_id]
    first_id = chunk["metadata"]["parent_record_id"]
    pending = [first_id]
    seen = set()
    context = []
    while pending:
        record_id = pending.pop(0)
        if record_id in seen:
            continue
        seen.add(record_id)
        record = parent_lookup[record_id]
        context.append(record)
        pending.extend(record["metadata"].get("required_context_record_ids", []))
    return context


# 이 예시는 벡터 검색이나 답변 생성이 아니라, 연결을 읽는 동작을 보여줍니다.
example_chunk = next(c for c in reviewed_search_chunks
    if c["metadata"]["parent_record_id"] == table_402_id)
example_context = read_reviewed_context(example_chunk["metadata"]["record_id"])
print("검색 조각 수:", len(reviewed_search_chunks))
print("표 조각을 찾았을 때 함께 읽는 기록:")
for record in example_context:
    print("-", record["metadata"]["title"], "| PDF", record["metadata"]["source_pages"])
print("최대 조각 길이:", max(c["metadata"]["token_count"] for c in reviewed_search_chunks), "토큰")


검색 조각 수: 2211
표 조각을 찾았을 때 함께 읽는 기록:
- 주행 모드별 패들 쉬프트의 기능 | PDF [402]
- 패들 쉬프트 (수동 변속 모드) | PDF [397]
- 회생 제동 시스템 사용 제한 | PDF [402]
- 드라이브 모드 별 패들 쉬프트 레버 기능 | PDF [398]
최대 조각 길이: 128 토큰


## 4. 보완 결과 확인

표 행이 통째로 남았는지, 원문 구간이 유지됐는지, 주의·경고가 부모 원문에 있는지 확인합니다.
이 확인은 이번 네 기록에 대한 것이며 다른 자동 초안의 정확성을 보장하지 않습니다.


In [5]:
# [프로젝트 추가 기록] 원본 화면에서 확인한 패들 쉬프트의 기호·표 행·그림을 보완합니다.
# 추가 이유·한계·수업 근거 전체 목록: src/zzong_santafe_lag/learning_additions.md
def collect_source_spans(records):
    """원문 구간을 정렬해 보완 전후의 누락과 중복을 비교합니다."""
    return sorted(
        (span["pdf_page_number"], span["start"], span["end"])
        for record in records for span in record["metadata"].get("source_spans", [])
    )


assert collect_source_spans(original_parents) == collect_source_spans(reviewed_parent_records)
assert len(parent_lookup) == len(reviewed_parent_records)
assert all(c["metadata"]["token_count"] <= token_budget for c in reviewed_search_chunks)
assert "î" in paddle_record["raw_text"] and "ï" in paddle_record["raw_text"]
assert not any(symbol in paddle_record["content"] for symbol in ["î", "ï"])
assert "[주의]" in paddle_record["content"]
assert "동시에 당기거나" in paddle_record["content"]
assert "[경고]" in limit_402_record["content"]
assert "브레이크 페달을 밟아 직접 속도를 조절하십시오." in limit_402_record["content"]
assert "SPORT 모드" in paddle_record["content"]
assert paddle_record["image_parts"][0]["pdf_image_key"] == "/I2"
assert table_398_record["metadata"]["table_rows"] == table_402_record["metadata"]["table_rows"]

# 각 부모 검색 글의 위치를 자식 조각이 연속해서 덮는지 확인합니다.
groups = defaultdict(list)
for chunk in reviewed_search_chunks:
    groups[chunk["metadata"]["parent_record_id"]].append(chunk)
for parent_id, parent in parent_lookup.items():
    cursor = 0
    for chunk in groups[parent_id]:
        assert chunk["metadata"]["parent_content_start"] == cursor
        cursor = chunk["metadata"]["parent_content_end"]
    assert cursor == len(parent["content"])

for table in [table_398_record, table_402_record]:
    # 네 표 행은 각각 어떤 한 조각 안에 완전한 형태로 있어야 합니다.
    for line in table["content"].splitlines():
        if line.startswith("드라이브 모드:"):
            assert any(line in c["content"] for c in groups[table["metadata"]["record_id"]])

replaced_ids = set(replacements)
for parent in original_parents:
    if parent["metadata"]["record_id"] not in replaced_ids:
        assert parent == parent_lookup[parent["metadata"]["record_id"]]

print("보완 전후 원문 구간:", "동일")
print("표 행:", "모드·기호·기능을 함께 보존")
print("397쪽 원문 기호:", "그대로 유지 / 검색용 글에서만 보완")
print("주의·경고:", "부모 원문과 연결된 확인 기록에 유지")
print("다른 원문 묶음:", "변경 없음")
print("상태별 기록 수:", dict(Counter(p["metadata"]["verification_status"]
    for p in reviewed_parent_records)))
print("전체 임베딩·데이터 파일·Supabase 저장:", "실행하지 않음")


보완 전후 원문 구간: 동일
표 행: 모드·기호·기능을 함께 보존
397쪽 원문 기호: 그대로 유지 / 검색용 글에서만 보완
주의·경고: 부모 원문과 연결된 확인 기록에 유지
다른 원문 묶음: 변경 없음
상태별 기록 수: {'sample_verified': 14, 'auto_draft_needs_review': 508, 'visually_reviewed_source': 4}
전체 임베딩·데이터 파일·Supabase 저장: 실행하지 않음


## 정리와 다음 순서

이번에 확인한 네 기록은 `visually_reviewed_source`로 표시했습니다.
이 표시는 페이지 대조를 마쳤다는 뜻이며, 임베딩 검색 성능이나 생성 답변의 정확성을 뜻하지 않습니다.
403쪽 이어지는 제한 경고는 초안으로 남겨 추가 원본 대조가 필요함을 표시했습니다.

`read_reviewed_context()`는 조각을 찾은 뒤 주의·제한까지 함께 읽는 예제입니다.
기존 08번 답변 실험은 아직 이 전체 자료에 연결하지 않았습니다.

다음은 PDF 43쪽의 **추천 오일 및 용량** 표입니다. 숫자·단위·추천 사양·각주 관계를 원본과 대조한 뒤,
검토 상태를 보존하는 전체 임베딩 실험과 .py 정리로 이어갑니다.


### 이번 실행 요약

- 전체 원문 묶음: 526개 / 검색 조각: 2,211개 / 최대 입력: 128토큰
- 이전 확인 예제 14개 + 이번 원본 대조 4개 + 검토가 필요한 자동 초안 508개
- 보완 전후 원문 구간 동일, 표 네 행 보존, 주의·경고와 해당 그림 연결 확인 완료
- 전체 임베딩과 데이터·DB 저장은 아직 실행하지 않았습니다.
